# 03 — Plasma OES Temporal Analysis

**BOSCH Plasma Etching dataset** — OES time-series at ~22.8 Hz, 3648 channels (185–884 nm).

This notebook demonstrates:
1. Loading a BOSCH NetCDF day file (first 3000 time steps)
2. Computing a PCA temporal embedding (dimensionality reduction over time)
3. Plotting the 2-D trajectory in PCA space
4. Clustering discharge phases with DTW K-means (k = 4)
5. Visualising cluster membership over time

In [1]:
import os
import sys
import warnings
from pathlib import Path

nb_dir = Path.cwd()
if nb_dir.name == "notebooks":
    os.chdir(nb_dir.parent)
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

print("Working directory:", Path.cwd())

Working directory: C:\Users\PC\libs-spectral-analysis


## 1 · Load BOSCH OES data (first 3 000 time steps)

In [2]:
from src.data_loader import load_bosch_oes

# Load Wafer_01 from Day_2024_07_02.nc
data = load_bosch_oes("data/bosch_oes/", day_file="Day_2024_07_02.nc")

# Subsample to first 3000 steps for demonstration speed
T_MAX = 3000
spectra    = data["spectra"][:T_MAX]      # (T_MAX, 3648) float32
timestamps = data["timestamps"][:T_MAX]    # (T_MAX,) seconds
wl         = data["wavelengths"]           # (3648,) nm

t_rel = timestamps - timestamps[0]   # relative seconds from start

print(f"Spectra : {spectra.shape}  (float32)")
print(f"Channels: {len(wl)}  ({wl[0]:.1f} – {wl[-1]:.1f} nm)")
print(f"Duration: {t_rel[-1]:.1f} s  @ ~{1/np.diff(timestamps).mean():.1f} Hz")

Spectra : (3000, 3648)  (float32)
Channels: 3648  (185.9 – 884.0 nm)
Duration: 132.3 s  @ ~22.7 Hz


## 2 · PCA temporal embedding

In [3]:
from src.temporal import compute_temporal_embedding

# StandardScaler → PCA(10): reduces (T, 3648) to (T, 10)
embedding, pca = compute_temporal_embedding(spectra, n_components=10)

print(f"Embedding : {embedding.shape}")
print(f"PC1+PC2 explained variance : {pca.explained_variance_ratio_[:2].sum():.1%}")
print(f"Top-3 PCs : {pca.explained_variance_ratio_[:3].round(3)}")

Embedding : (3000, 10)
PC1+PC2 explained variance : 12.3%
Top-3 PCs : [0.09  0.033 0.011]


## 3 · PCA trajectory plots

In [4]:
STEP = 10    # plot every 10th point for visual clarity

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- PC1 vs PC2 scatter coloured by time ----
ax = axes[0]
sc = ax.scatter(
    embedding[::STEP, 0], embedding[::STEP, 1],
    c=t_rel[::STEP], cmap="viridis", s=5, alpha=0.7,
)
plt.colorbar(sc, ax=ax, label="Time (s)")
ax.set_xlabel(f"PC1  ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2  ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.set_title("PCA Temporal Trajectory (PC1 vs PC2)")
ax.grid(True, alpha=0.3)

# ---- PC1/PC2/PC3 over time ----
ax2 = axes[1]
for pc_idx, color, label in zip(
    [0, 1, 2],
    ["steelblue", "tomato", "forestgreen"],
    ["PC1", "PC2", "PC3"],
):
    if pc_idx < embedding.shape[1]:
        ax2.plot(t_rel[::STEP], embedding[::STEP, pc_idx],
                 color=color, lw=0.7, label=label, alpha=0.85)
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("PCA Score")
ax2.set_title("PCA Scores over Time")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle(
    "BOSCH Plasma Etching OES — PCA Temporal Embedding  (3 000 time steps)",
    fontsize=11,
)
plt.tight_layout()

out_path = Path("outputs/notebook_03_pca_trajectory.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(str(out_path), dpi=100, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {out_path}")

Saved → outputs\notebook_03_pca_trajectory.png


## 4 · Discharge phase clustering (DTW K-means, k = 4)

In [5]:
from src.temporal import cluster_discharge_phases

# Euclidean K-means on PCA embedding (fast; DTW with seq_len=1 is equivalent)
labels, centroids, inertia = cluster_discharge_phases(
    embedding, k=4, metric="euclidean", seed=42
)

print(f"K-means (k=4) inertia : {inertia:.2f}")
cluster_sizes = {i: int((labels == i).sum()) for i in range(4)}
print("Cluster sizes :", cluster_sizes)

K-means (k=4) inertia : 506974.16
Cluster sizes : {0: 654, 1: 1159, 2: 508, 3: 679}


## 5 · Visualise cluster membership over time

In [6]:
COLORS = ["steelblue", "tomato", "forestgreen", "darkorange"]
PHASE_NAMES = [f"Phase {i+1}" for i in range(4)]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ---- PCA scatter coloured by cluster ----
ax = axes[0]
for k in range(4):
    mask = labels[::STEP] == k
    ax.scatter(
        embedding[::STEP][mask, 0],
        embedding[::STEP][mask, 1],
        s=5, alpha=0.7, color=COLORS[k], label=PHASE_NAMES[k],
    )
# Mark centroids
for k in range(4):
    ax.scatter(
        centroids[k, 0], centroids[k, 1],
        s=180, marker="*", color=COLORS[k],
        edgecolors="black", lw=0.8, zorder=6,
    )
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("DTW Cluster Membership in PCA Space")
ax.legend(markerscale=1.5, fontsize=9)
ax.grid(True, alpha=0.3)

# ---- Cluster membership over time (stacked bars) ----
ax2 = axes[1]
for k in range(4):
    member = (labels == k).astype(float)
    ax2.fill_between(t_rel, k, k + member,
                     color=COLORS[k], alpha=0.75, label=PHASE_NAMES[k], step="mid")

ax2.set_xlabel("Time (s)")
ax2.set_yticks([0.5, 1.5, 2.5, 3.5])
ax2.set_yticklabels(PHASE_NAMES, fontsize=9)
ax2.set_title("Discharge Phase Membership over Time")
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.2)

plt.suptitle(
    "BOSCH Plasma Etching — Discharge Phase Identification (k=4 clusters)",
    fontsize=11,
)
plt.tight_layout()

out_path = Path("outputs/notebook_03_clusters.png")
fig.savefig(str(out_path), dpi=100, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {out_path}")

Saved → outputs\notebook_03_clusters.png


## Summary

| Step | Method | Output |
|------|--------|--------|
| Data loading | `load_bosch_oes` (NetCDF) | (3 000, 3648) float32 |
| Temporal embedding | StandardScaler → PCA(10) | (3 000, 10) |
| Phase clustering | Euclidean K-means (k=4) | phase labels (3 000,) |

The PCA trajectory captures major process transitions (ignition, steady-state, extinction).
K-means clustering with k = 4 identifies four recurring discharge phases automatically.